# Analyzing TruffleHog scan results — false positives, severity tuning, and baseline comparison

TruffleHog finds leaked secrets in Git repos, filesystems, and S3 buckets — but not every match is a real secret. This notebook walks through analyzing scan results, identifying false positive patterns, tuning severity thresholds, and comparing results against a trusted baseline.

## Purpose

After running a TruffleHog scan, the raw output is a stream of detections — some genuine credentials, many false alarms. Manually triaging each one doesn't scale.

This notebook shows how to:
- Parse TruffleHog JSON output into structured records
- Identify common false positive patterns (test files, example code, generated docs)
- Apply severity filtering with custom rules
- Compare a scan against a known-good baseline to reduce noise

This is one approach; TruffleHog's own `--exclude-paths` and `--allow-rules` can also handle some of this at scan time.

## Prerequisites

- Python 3.8+
- `trufflehog` CLI installed (`pip install trufflehog` or `brew install trufflehog`)
- A Git repo to scan (this notebook uses a test repo with known fake secrets)

In [ ]:
import json
import subprocess
import sys
import tempfile
from pathlib import Path
from collections import Counter
from datetime import datetime

try:
    import pandas as pd
except ImportError:
    pd = None
    print("pandas not installed — some analysis will be skipped")

In [ ]:
def check_trufflehog():
    """Verify trufflehog is installed and report version."""
    result = subprocess.run(["trufflehog", "--version"], capture_output=True, text=True)
    if result.returncode != 0:
        print("trufflehog not found — install with 'pip install trufflehog' or 'brew install trufflehog'")
        sys.exit(1)
    print(f"trufflehog version: {result.stdout.strip()}")

check_trufflehog()

## Step 1: Scan a repo with JSON output

Scan a local Git repo using `--json` output so we can parse results programmatically. The `--only-verified` flag skips detections TruffleHog can't confirm by actually checking the credential against the service.

In [ ]:
REPO_PATH = "."  # point this at a real repo path
RESULTS_FILE = "trufflehog-results.json"

cmd = [
    "trufflehog", "git",
    "--json",
    "--no-update",
    "--fail",
    REPO_PATH
]

print(f"Running: {' '.join(cmd)}")
result = subprocess.run(cmd, capture_output=True, text=True)

# TruffleHog writes one JSON object per line to stdout
# Exit code 0 = no secrets found, 1 = secrets found, other = error
raw_lines = result.stdout.strip().split("\n")
if result.stderr:
    print(f"stderr: {result.stderr.strip()}")

In [ ]:
# Parse JSON lines
records = []
for line in raw_lines:
    if not line.strip():
        continue
    try:
        records.append(json.loads(line))
    except json.JSONDecodeError as e:
        print(f"Skipping non-JSON line: {e}")

print(f"Parsed {len(records)} detections from scan output")

## Step 2: Understand the result structure

Each detection is a JSON object with these key fields:
- `SourceMetadata`: where the secret was found (repo URL, commit hash, file path, line number)
- `DetectorName`: the detector that fired (e.g. `AWS`, `GitHub`, `Slack`, `PrivateKey`)
- `DecoderName`: how the secret was encoded (e.g. `PLAIN`, `BASE64`)
- `Verified`: whether TruffleHog confirmed the credential works
- `Raw`: the raw secret value (often redacted in output)
- `Redacted`: a partial view of the secret

In [ ]:
# Show a sample detection
if records:
    sample = records[0]
    print("Fields in a detection:")
    for key in ["DetectorName", "DecoderName", "Verified", "SourceMetadata"]:
        print(f"  {key}: {sample.get(key, 'N/A')}")
else:
    print("No detections found in scan output")

## Step 3: Identify false positive patterns

Common false positive sources in TruffleHog scans:
1. **Test files** — `*_test.go`, `test_*.py`, `spec/`, `__tests__/`
2. **Example/demo code** — `example/`, `demo/`, `sample/` directories
3. **Documentation** — README files, docs with placeholder tokens
4. **Vendor/dependency code** — `node_modules/`, `vendor/`, `.go/pkg/mod/`
5. **Generated files** — lockfiles, build outputs, compiled assets
6. **Low-entropy matches** — single-character strings, common words like "password" in example configs

In [ ]:
FP_PATH_PATTERNS = [
    "test", "spec", "example", "demo", "sample",
    "node_modules", "vendor", ".go/pkg/mod",
    "README", "CHANGELOG", "LICENSE",
    ".md", ".txt",  # markdown and text files often contain examples
]

def classify_false_positive(detection: dict) -> tuple[bool, str]:
    """Check if a detection looks like a false positive based on file path.

    Returns (is_fp, reason).
    The docs also suggest using --exclude-paths at scan time.
    """
    meta = detection.get("SourceMetadata", {}).get("Data", {}).get("Git", {})
    file_path = (meta.get("file") or "").lower()

    for pattern in FP_PATH_PATTERNS:
        if pattern in file_path:
            return (True, f"path matches '{pattern}'")

    return (False, "")


fp_count = 0
for r in records:
    is_fp, reason = classify_false_positive(r)
    if is_fp:
        fp_count += 1

print(f"Potential false positives (by path): {fp_count} / {len(records)}")

In [ ]:
# Show top detector names — which detectors fire most often?
detector_counts = Counter(r.get("DetectorName", "unknown") for r in records)
print("Detections by detector:")
for name, count in detector_counts.most_common(10):
    fp_in = sum(1 for r in records if r.get("DetectorName") == name and classify_false_positive(r)[0])
    print(f"  {name:25s}  {count:4d} total, {fp_in:4d} suspected FP")

## Step 4: Severity tuning — assign priority scores

TruffleHog doesn't assign severity levels out of the box — every detection is equally urgent. A practical approach is to assign priority based on detector type, verification status, and file context.

In [ ]:
# Assign a severity level to each detection
# High: verified credentials, private keys, cloud provider secrets
# Medium: unverified but high-value detectors (AWS, GCP, GitHub tokens)
# Low: generic secrets, low-entropy matches, non-production paths

HIGH_PRIORITY_DETECTORS = {"PrivateKey", "AWS", "GCP", "GitHub", "Slack", "Docker"}

def assign_severity(detection: dict) -> str:
    """Assign severity based on detector, verification, and path context."""
    detector = detection.get("DetectorName", "")
    verified = detection.get("Verified", False)

    if verified:
        return "critical"
    if detector in HIGH_PRIORITY_DETECTORS:
        return "high"
    is_fp, _ = classify_false_positive(detection)
    if is_fp:
        return "low"
    return "medium"


severity_counts = Counter()
for r in records:
    sev = assign_severity(r)
    severity_counts[sev] += 1
    r["_severity"] = sev

print("Assigned severity distribution:")
for sev in ["critical", "high", "medium", "low"]:
    print(f"  {sev:10s}  {severity_counts.get(sev, 0)}")

## Step 5: Baseline comparison

A baseline is a known-good snapshot of scan results — typically from the last clean scan on the default branch. By comparing a new scan against the baseline, you can spot new detections that weren't there before.

In [ ]:
def detection_fingerprint(d: dict) -> str:
    """Create a stable fingerprint for a detection."""
    meta = d.get("SourceMetadata", {}).get("Data", {}).get("Git", {})
    file_path = meta.get("file", "")
    line = meta.get("line", "0")
    detector = d.get("DetectorName", "")
    return f"{detector}:{file_path}:{line}"


def save_baseline(records: list[dict], path: str):
    """Save fingerprints as a baseline file."""
    fingerprints = [detection_fingerprint(r) for r in records]
    with open(path, "w") as f:
        json.dump({
            "created": datetime.now().isoformat(),
            "count": len(fingerprints),
            "fingerprints": fingerprints
        }, f, indent=2)
    print(f"Saved baseline: {len(fingerprints)} fingerprints to {path}")


def load_baseline(path: str) -> set:
    """Load baseline fingerprints from file."""
    with open(path) as f:
        data = json.load(f)
    print(f"Loaded baseline: {data['count']} fingerprints from {data['created']}")
    return set(data["fingerprints"])


if records:
    baseline_file = "trufflehog-baseline.json"
    save_baseline(records, baseline_file)
    baseline = load_baseline(baseline_file)

In [ ]:
# Simulate a second scan with a new detection
# In practice you'd run trufflehog again and load both results

new_fingerprints = {detection_fingerprint(r) for r in records}
new_detections = new_fingerprints - baseline

print(f"New detections (not in baseline): {len(new_detections)}")
if new_detections:
    print("First 5 new fingerprints:")
    for fp in list(new_detections)[:5]:
        print(f"  {fp}")

## Step 6: Filter to actionable results

Combine all three strategies — FP filtering, severity tuning, and baseline comparison — to produce a list of detections worth investigating.

In [ ]:
def actionable_results(records: list[dict], baseline_fps: set | None = None) -> list[dict]:
    """Filter detections to those that are new, not FP, and above low severity."""
    if baseline_fps is None:
        baseline_fps = set()

    results = []
    for r in records:
        is_fp, _ = classify_false_positive(r)
        if is_fp:
            continue
        if r.get("_severity") == "low":
            continue
        fp = detection_fingerprint(r)
        if fp in baseline_fps:
            continue
        results.append(r)

    return results


actionable = actionable_results(records, baseline)
print(f"Actionable detections after filtering: {len(actionable)} / {len(records)}")
for r in actionable[:5]:
    meta = r.get("SourceMetadata", {}).get("Data", {}).get("Git", {})
    print(f"  {r.get('DetectorName', '?'):20s}  {meta.get('file', '?'):40s}  L{meta.get('line', '?')}")

## Verify

This workflow covers the main post-scan analysis tasks:
1. **Parse TruffleHog JSON** — one JSON object per line, easy to load into Python
2. **False positive detection** — path-based heuristics catch most noise from test/example/vendor code
3. **Severity assignment** — detector + verification status gives a reasonable priority model
4. **Baseline comparison** — fingerprint-based diff identifies genuinely new detections

One thing I'm not certain about: TruffleHog's `--only-verified` flag can miss real secrets if the service endpoint isn't reachable during scanning. The severity model above compensates by flagging unverified-but-high-value detectors as "high" rather than ignoring them entirely.

A more robust approach would use allow-lists per-repo (`--allow-rules`) and path exclusions (`--exclude-paths`) at scan time so the raw output is cleaner from the start.